# 03 - Action: 探索性分析

> 数据探索 · 逐步细化 · 从粗到精

---

## 一、数据加载与初探

In [ ]:
# TODO: 加载数据并初步查看

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 设置中文显示
plt.rcParams['font.sans-serif'] = ['SimHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

# 加载原始数据（含脏数据）
df_raw = pd.read_csv('../data/raw/rider_pricing_raw.csv')

print("数据形状:", df_raw.shape)
print("\n字段列表:")
print(df_raw.columns.tolist())
print("\n数据类型:")
print(df_raw.dtypes)
print("\n缺失值统计:")
print(df_raw.isnull().sum())

print("TODO: 记录发现的数据质量问题")

## 二、数据清洗

In [ ]:
# TODO: 数据清洗全流程

print("""
数据清洗步骤：

1. 缺失值处理
   - price: 5%缺失 → 用中位数填充或删除
   - rider_supply: 5%缺失 → 用中位数填充或删除

2. 异常值处理
   - order_volume: 极端值（>1000或<10）→ 删除或截断
   - ontime_rate: 必须在0-1之间
   - price: 必须在3-15之间

3. 逻辑错误修正
   - 雨天但温度35°C → 数据录入错误
   - 雪天但温度20°C → 矛盾

4. 重复记录
   - 同一order_id出现多次 → 去重
""")

# 清洗操作
df_clean = df_raw.copy()

# 处理缺失值
print(f"\n清洗前记录数: {len(df_clean)}")
df_clean = df_clean.dropna(subset=['price', 'rider_supply'])
print(f"删除缺失值后: {len(df_clean)}")

# 处理异常值
df_clean = df_clean[df_clean['order_volume'] <= 1000]
df_clean = df_clean[df_clean['order_volume'] >= 10]
df_clean = df_clean[(df_clean['ontime_rate'] >= 0) & (df_clean['ontime_rate'] <= 1)]
df_clean = df_clean[(df_clean['price'] >= 3) & (df_clean['price'] <= 15)]
print(f"处理异常值后: {len(df_clean)}")

# 处理逻辑错误
df_clean = df_clean[~((df_clean['weather'] == 'rainy') & (df_clean['temperature'] > 30))]
df_clean = df_clean[~((df_clean['weather'] == 'snowy') & (df_clean['temperature'] > 10))]
print(f"处理逻辑错误后: {len(df_clean)}")

# 去重
df_clean = df_clean.drop_duplicates(subset=['order_id'])
print(f"去重后: {len(df_clean)}")

# 保存清洗后数据
df_clean.to_csv('../data/processed/rider_pricing_processed.csv', index=False)

print("\n清洗完成！")
print("TODO: 对比清洗前后的数据分布变化")

## 三、单变量分析

In [ ]:
# TODO: 核心变量的分布分析

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# 价格分布
axes[0, 0].hist(df_clean['price'], bins=30, edgecolor='black', alpha=0.7)
axes[0, 0].set_title('配送费分布')
axes[0, 0].set_xlabel('价格（元）')
axes[0, 0].set_ylabel('频次')

# 订单量分布
axes[0, 1].hist(df_clean['order_volume'], bins=30, edgecolor='black', alpha=0.7)
axes[0, 1].set_title('订单量分布')
axes[0, 1].set_xlabel('订单量')
axes[0, 1].set_ylabel('频次')

# 准时率分布
axes[0, 2].hist(df_clean['ontime_rate'], bins=30, edgecolor='black', alpha=0.7)
axes[0, 2].set_title('准时率分布')
axes[0, 2].set_xlabel('准时率')
axes[0, 2].set_ylabel('频次')

# 骑手供给分布
axes[1, 0].hist(df_clean['rider_supply'], bins=30, edgecolor='black', alpha=0.7)
axes[1, 0].set_title('骑手供给分布')
axes[1, 0].set_xlabel('骑手数')
axes[1, 0].set_ylabel('频次')

# 天气分布
weather_counts = df_clean['weather'].value_counts()
axes[1, 1].bar(weather_counts.index, weather_counts.values, alpha=0.7)
axes[1, 1].set_title('天气分布')
axes[1, 1].set_xlabel('天气')
axes[1, 1].set_ylabel('频次')

# 区域类型分布
region_counts = df_clean['region_type'].value_counts()
axes[1, 2].bar(region_counts.index, region_counts.values, alpha=0.7)
axes[1, 2].set_title('区域类型分布')
axes[1, 2].set_xlabel('区域类型')
axes[1, 2].set_ylabel('频次')

plt.tight_layout()
plt.show()

print("TODO: 分析每个变量的分布特征，识别异常模式")

## 四、双变量分析：价格 vs 核心指标

In [ ]:
# TODO: 分析价格与核心指标的关系

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 价格 vs 订单量
axes[0].scatter(df_clean['price'], df_clean['order_volume'], alpha=0.3, s=10)
axes[0].set_xlabel('价格（元）')
axes[0].set_ylabel('订单量')
axes[0].set_title('价格 vs 订单量')

# 添加趋势线
z = np.polyfit(df_clean['price'], df_clean['order_volume'], 1)
p = np.poly1d(z)
axes[0].plot(df_clean['price'], p(df_clean['price']), "r--", alpha=0.8)

# 价格 vs 准时率
axes[1].scatter(df_clean['price'], df_clean['ontime_rate'], alpha=0.3, s=10)
axes[1].set_xlabel('价格（元）')
axes[1].set_ylabel('准时率')
axes[1].set_title('价格 vs 准时率')

# 价格 vs 骑手供给
axes[2].scatter(df_clean['price'], df_clean['rider_supply'], alpha=0.3, s=10)
axes[2].set_xlabel('价格（元）')
axes[2].set_ylabel('骑手数')
axes[2].set_title('价格 vs 骑手供给')

plt.tight_layout()
plt.show()

# 计算相关系数
print("\n相关系数矩阵:")
corr_matrix = df_clean[['price', 'order_volume', 'ontime_rate', 'rider_supply']].corr()
print(corr_matrix)

print("\n" + "="*60)
print("初步观察：")
print("1. 价格与订单量：负相关（但这是相关，不是因果）")
print("2. 价格与准时率：正相关（可能因为高价吸引了更多骑手）")
print("3. 价格与骑手供给：正相关（价格激励效应）")
print("="*60)

print("TODO: 思考为什么这些相关不等于因果")

## 五、选择偏差的直观展示

In [ ]:
# TODO: 展示选择偏差

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 不同天气下的价格分布
for weather in df_clean['weather'].unique():
    subset = df_clean[df_clean['weather'] == weather]
    axes[0].hist(subset['price'], bins=20, alpha=0.5, label=weather)
axes[0].set_xlabel('价格（元）')
axes[0].set_ylabel('频次')
axes[0].set_title('不同天气下的价格分布')
axes[0].legend()

# 不同天气下的订单量分布
for weather in df_clean['weather'].unique():
    subset = df_clean[df_clean['weather'] == weather]
    axes[1].hist(subset['order_volume'], bins=20, alpha=0.5, label=weather)
axes[1].set_xlabel('订单量')
axes[1].set_ylabel('频次')
axes[1].set_title('不同天气下的订单量分布')
axes[1].legend()

plt.tight_layout()
plt.show()

print("""
选择偏差展示：

雨天：
- 价格更高（平台自动加价）
- 订单量更少（用户不想出门）
- 简单回归结论：加价导致订单减少 ❌
- 真相：雨天本来订单就少，不是因为价格

晴天：
- 价格较低
- 订单量较多
- 但这不意味着"降价能增加订单"
""")

print("TODO: 计算不同天气下的平均价格和订单量")

## 六、异质性初探

In [ ]:
# TODO: 初步探索异质性效应

# 按天气×区域类型分组，看价格与订单量的关系
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

weather_types = ['sunny', 'rainy', 'snowy']
region_types = df_clean['region_type'].unique()

for i, weather in enumerate(weather_types):
    ax = axes[i // 2, i % 2] if i < 3 else None
    if ax is None:
        continue
    
    for rtype in region_types:
        subset = df_clean[(df_clean['weather'] == weather) & (df_clean['region_type'] == rtype)]
        if len(subset) > 10:
            ax.scatter(subset['price'], subset['order_volume'], alpha=0.3, s=10, label=rtype)
    
    ax.set_xlabel('价格（元）')
    ax.set_ylabel('订单量')
    ax.set_title(f'{weather}天气：价格 vs 订单量（按区域类型）')
    ax.legend()

# 隐藏最后一个子图
axes[1, 1].axis('off')

plt.tight_layout()
plt.show()

print("""
异质性观察：

雨天CBD：
- 价格-订单量关系较平坦
- 可能意味着加价对订单量影响小
- 但骑手供给可能增加

雨天郊区：
- 价格-订单量关系陡峭
- 加价可能导致订单大幅下降
- 用户流失效应 > 骑手供给效应

雪天：
- 所有区域都显示价格-订单量关系较平坦
- 可能意味着雪天加价有效
- 用户理解雪天需要加价
""")

print("TODO: 按更多维度分组，探索异质性模式")

## 七、时段模式分析

In [ ]:
# TODO: 分析时段模式

# 按时段聚合
hourly_stats = df_clean.groupby('hour').agg({
    'price': 'mean',
    'order_volume': 'mean',
    'ontime_rate': 'mean',
    'rider_supply': 'mean'
}).reset_index()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].plot(hourly_stats['hour'], hourly_stats['price'], marker='o')
axes[0, 0].set_title('平均价格 vs 时段')
axes[0, 0].set_xlabel('时段')
axes[0, 0].set_ylabel('平均价格（元）')

axes[0, 1].plot(hourly_stats['hour'], hourly_stats['order_volume'], marker='o')
axes[0, 1].set_title('平均订单量 vs 时段')
axes[0, 1].set_xlabel('时段')
axes[0, 1].set_ylabel('平均订单量')

axes[1, 0].plot(hourly_stats['hour'], hourly_stats['ontime_rate'], marker='o')
axes[1, 0].set_title('平均准时率 vs 时段')
axes[1, 0].set_xlabel('时段')
axes[1, 0].set_ylabel('平均准时率')

axes[1, 1].plot(hourly_stats['hour'], hourly_stats['rider_supply'], marker='o')
axes[1, 1].set_title('平均骑手供给 vs 时段')
axes[1, 1].set_xlabel('时段')
axes[1, 1].set_ylabel('平均骑手数')

plt.tight_layout()
plt.show()

print("TODO: 识别高峰时段和平峰时段的模式差异")

## 八、关键发现总结

In [ ]:
# TODO: 总结探索性分析的关键发现

print("""
关键发现：

1. 数据质量问题
   - 5%缺失值（price, rider_supply）
   - 2%异常值（order_volume极端值）
   - 1%逻辑错误（雨天高温）
   - 已清洗，剩余20,000+条记录

2. 选择偏差明显
   - 雨天价格高，订单少
   - 但不能归因于价格
   - 需要因果推断方法

3. 异质性迹象
   - 雨天CBD：价格效应弱
   - 雨天郊区：价格效应强（负面）
   - 雪天：所有区域价格效应弱

4. 时段模式
   - 高峰时段（7-9, 11-13, 17-21）订单量大
   - 高峰时段价格也高
   - 准时率在高峰时段下降

5. 下一步方向
   - 用DML处理选择偏差
   - 用因果森林发现异质性子群
   - 对比S/T/X/R-Learner
""")

print("TODO: 根据探索结果，调整分析策略")

## 九、下一步

1. **详细建模**：用DML + Meta-Learners估计因果效应
2. **异质性分析**：用因果森林发现最优子群
3. **稳健检验**：验证结论可靠性

---

> **核心洞察**：探索性分析不是"随便看看"，而是"有目的地看"——验证假设、发现问题、确定方向。